In [1]:
# -----------------------------------------------------------------------------
# Import libraries
# - numpy: numerical operations (arrays, math functions)
# - pandas: tabular data manipulation, CSV IO, datetime handling
# These libraries are fundamental for cleaning and analyzing the bike trip data.
# -----------------------------------------------------------------------------

# Import required libraries for data manipulation
import numpy as np
import pandas as pd


In [2]:
# -----------------------------------------------------------------------------
# Load dataset
# This loads the cleaned bike trip CSV covering July 2025 to June 2026 into a pandas DataFrame named 'df'.
# Expected columns include: trip_id, bike_type, started_at, ended_at, start_station_name, end_station_name, member_casual, start_lat, start_lng, end_lat, end_lng
# Replace the filename here if your CSV is in a different path or has a different name.
# -----------------------------------------------------------------------------

# Load the cleaned bike trip dataset for the period of July 2025 to June 2026
df = pd.read_csv('bike_trip_data_clean(July 2025-June 2026).csv')


In [3]:
# -----------------------------------------------------------------------------
# Convert column data types for memory and correctness
# - trip_id -> string to ensure IDs are treated as identifiers, not numeric
# - bike_type, start_station_name, end_station_name, member_casual -> category to save memory when there are many repeated values
# This helps performance on large datasets and clarifies downstream operations.
# -----------------------------------------------------------------------------

# Convert text/categorical columns to 'category' type to save memory
df = df.astype({
    'trip_id': 'str',
    'bike_type': 'category',
    'start_station_name': 'category',
    'end_station_name': 'category',
    'member_casual' : 'category'
    })


In [4]:
# -----------------------------------------------------------------------------
# Convert time columns to pandas datetime objects
# This enables vectorized datetime operations (differences, extraction of components, resampling).
# -----------------------------------------------------------------------------

# Convert time string columns format to standard Pandas datetime objects
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])


In [5]:
# -----------------------------------------------------------------------------
# Create duration and date-related features
# - gap: raw timedelta between ended_at and started_at
# - duration_minutes: numeric total minutes of each trip (rounded to 2 decimals)
# - started_date: date-only value derived from started_at (time component removed)
# - rental_days: day name (Monday..Sunday) for weekday/weekend analysis
# Note: started_date is temporarily converted to string then back to datetime to normalize the column to date-only.
# -----------------------------------------------------------------------------

# Calculate total duration between trip end and start time
df['gap'] = df['ended_at'] - df['started_at']

# Convert time gap to total minutes, rounded to 2 decimal places
df['duration_minutes'] = round(df['gap']/pd.Timedelta(minutes=1),2)

# Extract day of the week (e.g., 'Monday', 'Tuesday')
df['started_date'] = df['started_at'].dt.strftime('%B %d, %Y')

# Extract date without timestamp
df['rental_days'] = df['started_at'].dt.day_name()

# Convert date string columns format to standard Pandas datetime objects
df['started_date'] = pd.to_datetime(df['started_date'])


In [6]:
# -----------------------------------------------------------------------------
# Drop unneeded columns
# Remove latitude/longitude columns and the temporary 'gap' column to reduce memory usage for subsequent analysis.
# If spatial analysis is needed later, keep the coordinate columns instead of dropping them.
# -----------------------------------------------------------------------------

# Drop unnecessary location coordinates and the temporary time gap column
df = df.drop(columns=['start_lat', 'start_lng', 'end_lat', 'end_lng', 'gap'])


In [7]:
# -----------------------------------------------------------------------------
# Display DataFrame sample
# Show the DataFrame (Jupyter will render a table) so the user can visually inspect columns, types, and sample rows.
# -----------------------------------------------------------------------------

# Display the updated DataFrame
df


,trip_id,bike_type,started_at,ended_at,start_station_name,end_station_name,member_casual,duration_minutes,started_date,rental_days
0,455D43BD91D73437,classic_bike,2025-07-05 17:15:08.456,2025-07-05 17:25:47.079,lincoln ave & diversey pkwy,lincoln ave & addison st,member,10.64,2025-07-05,Saturday
1,9D4A6B723ECD98CA,classic_bike,2025-07-01 13:57:38.878,2025-07-01 14:06:35.780,cottage grove ave & oakwood blvd,cottage grove ave & 47th st,member,8.95,2025-07-01,Tuesday
2,C57044CF523302ED,classic_bike,2025-07-31 16:49:28.142,2025-07-31 17:15:28.999,theater on the lake,winthrop ave & lawrence ave,member,26.01,2025-07-31,Thursday
3,AFD35552E6685B6E,electric_bike,2025-07-17 09:36:21.058,2025-07-17 09:46:54.706,pine grove ave & waveland ave,winthrop ave & lawrence ave,member,10.56,2025-07-17,Thursday
4,C0582EBAA6CED519,classic_bike,2025-07-02 18:43:45.213,2025-07-02 18:57:06.687,theater on the lake,sheffield ave & wellington ave,member,13.36,2025-07-02,Wednesday
...,...,...,...,...,...,...,...,...,...,...
5926397,86932262232F29F0,electric_bike,2026-06-17 21:09:10.026,2026-06-17 21:15:09.708,clark st & lincoln ave,stockton dr & wrightwood ave,member,5.99,2026-06-17,Wednesday
5926398,6654F4B0FC4D361E,classic_bike,2026-06-07 08:07:09.330,2026-06-07 08:09:11.320,california ave & cortez st,california ave & division st,member,2.03,2026-06-07,Sunday
5926399,6A71125A045366A8,electric_bike,2026-06-02 15:36:13.487,2026-06-02 15:47:10.902,monticello ave & diversey ave,troy st & grace st,member,10.96,2026-06-02,Tuesday
5926400,152C05EEB9316CD6,electric_bike,2026-06-10 15:55:03.821,2026-06-10 16:03:57.409,leavitt st & addison st,troy st & grace st,member,8.89,2026-06-10,Wednesday


In [30]:
# -----------------------------------------------------------------------------
# Summary statistics
# Use df.describe() to inspect count, mean, percentiles and min/max for numeric and datetime columns.
# This helps identify anomalies, ranges, and potential data quality issues.
# -----------------------------------------------------------------------------

df.describe().round(2)


,started_at,ended_at,duration_minutes,started_date
count,5926402,5926402,5926402.00,5926402
mean,2025-12-17 19:20:11.809314,2025-12-17 19:34:36.455872,14.41,2025-12-17 04:49:37.118325
min,2025-06-30 15:25:11.437000,2025-07-01 00:00:02.027000,0.00,2025-06-30 00:00:00
25%,2025-08-29 13:32:22.879500,2025-08-29 13:49:43.756000,5.36,2025-08-29 00:00:00
50%,2025-11-04 09:18:24.020500,2025-11-04 09:29:34.961500,9.36,2025-11-04 00:00:00
75%,2026-04-26 12:13:28.767250,2026-04-26 12:28:47.118500,16.44,2026-04-26 00:00:00
max,2026-06-30 23:58:47.113000,2026-06-30 23:59:55.118000,1499.97,2026-06-30 00:00:00
std,NaN,NaN,29.22,NaN


In [ ]:
# -----------------------------------------------------------------------------
# Outlier removal (duration_minutes)
# - Remove trips shorter than 1 minute as likely noise or artifacts.
# - Use the IQR method to detect upper outliers: outlier_line = Q3 + 1.5 * IQR.
# - Filter out trips with duration_minutes greater than the calculated outlier_line.
# This keeps the analysis focused on typical trip durations and reduces skew from extreme values.
# -----------------------------------------------------------------------------


check = df['duration_minutes']>=1
df = df[check]

q1 = df['duration_minutes'].quantile(0.25)
q3 = df['duration_minutes'].quantile(0.75)
iqr = q3 - q1

outlier_line = q3 + (1.5 * iqr)

df= df[df['duration_minutes']<= outlier_line]


In [63]:
# -----------------------------------------------------------------------------
# Summary statistics
# Use df.describe() to inspect count, mean, percentiles and min/max for numeric and datetime columns.
# This helps identify anomalies, ranges, and potential data quality issues.
# -----------------------------------------------------------------------------

df.describe().round(2)


,started_at,ended_at,duration_minutes,started_date
count,5355968,5355968,5355968.00,5355968
mean,2025-12-18 17:25:27.973383,2025-12-18 17:36:20.188539,10.87,2025-12-18 02:56:01.688794
min,2025-06-30 23:33:38.273000,2025-07-01 00:00:02.027000,1.00,2025-06-30 00:00:00
25%,2025-08-30 18:39:20.492000,2025-08-30 18:52:41.632250,5.40,2025-08-30 00:00:00
50%,2025-11-06 01:02:25.252000,2025-11-06 01:12:09.334000,8.97,2025-11-06 00:00:00
75%,2026-04-25 18:34:22.348000,2026-04-25 18:45:19.365750,14.65,2026-04-25 00:00:00
max,2026-06-30 23:56:32.103000,2026-06-30 23:59:55.118000,33.40,2026-06-30 00:00:00
std,NaN,NaN,7.13,NaN


In [65]:
df

,trip_id,bike_type,started_at,ended_at,start_station_name,end_station_name,member_casual,duration_minutes,started_date,rental_days
0,455D43BD91D73437,classic_bike,2025-07-05 17:15:08.456,2025-07-05 17:25:47.079,lincoln ave & diversey pkwy,lincoln ave & addison st,member,10.64,2025-07-05,Saturday
1,9D4A6B723ECD98CA,classic_bike,2025-07-01 13:57:38.878,2025-07-01 14:06:35.780,cottage grove ave & oakwood blvd,cottage grove ave & 47th st,member,8.95,2025-07-01,Tuesday
2,C57044CF523302ED,classic_bike,2025-07-31 16:49:28.142,2025-07-31 17:15:28.999,theater on the lake,winthrop ave & lawrence ave,member,26.01,2025-07-31,Thursday
3,AFD35552E6685B6E,electric_bike,2025-07-17 09:36:21.058,2025-07-17 09:46:54.706,pine grove ave & waveland ave,winthrop ave & lawrence ave,member,10.56,2025-07-17,Thursday
4,C0582EBAA6CED519,classic_bike,2025-07-02 18:43:45.213,2025-07-02 18:57:06.687,theater on the lake,sheffield ave & wellington ave,member,13.36,2025-07-02,Wednesday
...,...,...,...,...,...,...,...,...,...,...
5926397,86932262232F29F0,electric_bike,2026-06-17 21:09:10.026,2026-06-17 21:15:09.708,clark st & lincoln ave,stockton dr & wrightwood ave,member,5.99,2026-06-17,Wednesday
5926398,6654F4B0FC4D361E,classic_bike,2026-06-07 08:07:09.330,2026-06-07 08:09:11.320,california ave & cortez st,california ave & division st,member,2.03,2026-06-07,Sunday
5926399,6A71125A045366A8,electric_bike,2026-06-02 15:36:13.487,2026-06-02 15:47:10.902,monticello ave & diversey ave,troy st & grace st,member,10.96,2026-06-02,Tuesday
5926400,152C05EEB9316CD6,electric_bike,2026-06-10 15:55:03.821,2026-06-10 16:03:57.409,leavitt st & addison st,troy st & grace st,member,8.89,2026-06-10,Wednesday


## Insight

### 1. Temporal Usage Pattern

This analysis aims to test the hypothesis that Members use bikes for productivity/commuting, while Casuals use them for recreation.

**Insight sought:**
Which day of the week (rental_days) is the busiest for each user group?

**How to interpret the insight:**
If Members peak on Monday?Friday (workdays) and Casuals peak on Saturday?Sunday (weekends), then their behavioral patterns differ functionally.

### 2. Bike Type Preference (Product Preference)

This analysis examines whether membership status affects the choice of bike technology (bike_type).

**Insight sought:**
Do Members prefer electric_bike for time efficiency, or do they prefer classic_bike?

**How to interpret the insight:**
If one group strongly favors electric bikes, that has operational implications (for example, battery allocation and fleet management).

### 3. Analysis of Popular Stations (Spatial Behavior)

Identify hubs and main routes for each user group (start_station_name and end_station_name).

**Insight sought:**
Top 5 departure stations for Casual vs Member.

**How to interpret the insight:**
Casual users are typically concentrated near tourist attractions, parks, or waterfront areas (e.g., Theater on the Lake). Members are typically concentrated near major transit stations, bus stops, or business districts.

### 4. Trip Duration Analysis (Trip Duration Behavior)

Although df.describe() has already been run, further analysis comparing average trip duration by weekday can be informative.

**Insight sought:**
Do Members' trip durations lengthen on weekends like Casuals, or do they remain short?


In [ ]:
# -----------------------------------------------------------------------------
# Prepare weekday ordering and compute counts by day & membership status
# - Convert rental_days to a categorical with an explicit order so groupby results are sorted Monday-Sunday.
# - Group by rental_days and member_casual to get the usage pattern across weekdays/weekends.
# -----------------------------------------------------------------------------

days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

df['rental_days'] = pd.Categorical(df['rental_days'], categories=days_order, ordered=True)

days_pattern = df.groupby(['rental_days', 'member_casual'], observed=False).size().unstack().sort_values('rental_days')
days_pattern


member_casual,casual,member
rental_days,,
Monday,205436,503313
Tuesday,213771,584595
Wednesday,207740,570780
Thursday,230943,574908
Friday,280508,525327
Saturday,361154,445257
Sunday,278423,373813


In [75]:
# -----------------------------------------------------------------------------
# Bike type preference by user group
# Compute normalized counts (percentages) of bike_type within each member_casual group to see preference for electric vs classic bikes.
# -----------------------------------------------------------------------------

bike_pattern = df.groupby('member_casual')['bike_type'].value_counts(normalize=True).unstack() * 100
print(bike_pattern.round(2))


bike_type      classic_bike  electric_bike
member_casual                             
casual                27.55          72.45
member                33.68          66.32


In [ ]:
# -----------------------------------------------------------------------------
# Top start stations per user group
# Print top-5 most frequent start_station_name for Casual and for Member separately ? useful to identify hotspots for each user segment.
# -----------------------------------------------------------------------------

print("Top 5 Casual Stations:")
print(df[df['member_casual'] == 'casual']['start_station_name'].value_counts().head(5))

print("\nTop 5 Member Stations:")
print(df[df['member_casual'] == 'member']['start_station_name'].value_counts().head(5))


Top 5 Casual Stations:
start_station_name
navy pier                             40273
dusable lake shore dr & monroe st     30589
dusable lake shore dr & north blvd    21387
clark st & schiller st                19578
theater on the lake                   18016
Name: count, dtype: int64

Top 5 Member Stations:
start_station_name
canal st & adams st            44337
dearborn pkwy & delaware pl    38101
kingsbury st & kinzie st       35689
clark st & schiller st         31950
new st & illinois st           31615
Name: count, dtype: int64


In [ ]:
# -----------------------------------------------------------------------------
# Average trip duration by weekday and user group
# Group by rental_days and member_casual then compute mean duration_minutes to compare typical trip lengths across days and segments.
# -----------------------------------------------------------------------------

durasi_harian = df.groupby(['rental_days', 'member_casual'])['duration_minutes'].mean().unstack()
print(durasi_harian.round(2))


member_casual  casual  member
rental_days                  
Monday          11.78   10.03
Tuesday         11.19   10.12
Wednesday       11.04   10.10
Thursday        11.23   10.16
Friday          11.98   10.18
Saturday        13.13   10.93
Sunday          12.99   10.74
